# Indholdsfortegnelse

- [BBR Bitemporalitet](#BBR_Bitemporalitet)

  - Her forklares omkring strukturen i det bitemporale (historiske) BBR 


- [Filtrering](#Filtrering)

  - Her anvendes filtreringen på det rå BBR. Outputtet er det relevante BBR som ligger til grund for alt arbejde (altså det rå BBR bruges ikke til noget)

- [Sortering (hvis det viser sig nyttigt)](#Sortering)

  - Her dannes mindre filer der repræsentere årlige snapshots af BBR på en given 1. juli (midt i året og i overensstemmelse med resterende code i pipelinen, hvor jeg definerer en lokalebasen bygning som værende fra 1. juli YYYY). Fx 1. juli 2026

<a id="BBR_Bitemporalitet"></a>

## BBR Bitemporalitet

Dette er en forklaring på hvordan man sorterer i det historiske BBR datasæt, hvor alle tidligere registreringer fremgår - også dem der ikke længere er relevante.

Vi ønsker kun at beholde de observationer som stadig er "registreret" i BBR. Altså fjerne obserbationer som ikke længere er en del af datasættet "registreret". Dette er en teknikalitet hvor fx et hus registreres med virkning fra 2016 - NaT. Denne række slettes ikke når bygningens andevelse fx ændres i 2018, her vil rækken blot ikke længere være "registreret".


### Eksempel på hvordan en bygnings nye anvendelse registreres i BBR:
```text
  registreretFra    registreretTil   virkningFra    VirkningTil      id       anvendelse
     2016               NaT            2016             NaT         123          10                      bygning 123 er registreret med anvendelse 10 fra 2016 til ubestemt tid

  registreretFra    registreretTil   virkningFra    VirkningTil      id       anvendelse
     2016               2018           2016             NaT         123          10                      bygning 123 er fra 2016 til 2018 registreret som anvendelse 10 fra 2016 til ubestemt tid (forrige registrering udgår)
     2018               NaT            2016            2018         123          20                      bygning 123 står registreret fra 2018 på ubestemt tid som anvendelse 10 i perioden 2016-2018 (arkiv over historikken)
     2018               NaT            2018             NaT         123          20                      bygning 123 er nu (på ubestemt tid) registreret som anvendelse = 20
```


altså skal vi blot sorterer på registreret til = NaT (Python betingelsen: registreringTil.isna())

## Filtrering

In [1]:
# indlæs pakker
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from pandas.errors import DtypeWarning
import warnings
warnings.filterwarnings("ignore", category=DtypeWarning)

In [ ]:
"""Chunked BBR corrections using pandas and temporary CSV files.
Dependencies: pandas, tqdm. No database or SQL.
RAM is limited to working copies of a chunk or one unusually large building history.
"""

from pathlib import Path
from tempfile import TemporaryDirectory
import pandas as pd
from tqdm.auto import tqdm

UDFASEDE_ANVENDELSESKODER = {
    130, 210, 220, 230, 290, 310, 320, 330,
    390, 410, 420, 430, 440, 490, 520, 530,
}


def _ret_bygning(
    rows,
    history,
    overwrite_coordinate,
    overwrite_construction_year,
    overwrite_main_use,
):
    """Correct one chunk's building rows using that building's full history."""
    out = rows.copy()
    dates = pd.to_datetime(
        history["virkningFra"], utc=True, format="mixed", errors="coerce"
    )

    # 1. Coordinates: closest date with strictly better quality.
    # Use original donor rows, so a correction cannot become another donor.
    if overwrite_coordinate:
        quality = pd.to_numeric(
            history["byg134KvalitetAfKoordinatsæt"], errors="coerce"
        )
        donors = history.loc[
            dates.notna()
            & quality.notna()
            & history["byg404Koordinat"].notna()
        ].copy()
        donors["_date"] = dates.loc[donors.index]
        donors["_quality"] = quality.loc[donors.index]

        row_dates = pd.to_datetime(
            rows["virkningFra"], utc=True, format="mixed", errors="coerce"
        )
        row_quality = pd.to_numeric(
            rows["byg134KvalitetAfKoordinatsæt"], errors="coerce"
        )

        for idx in rows.index:
            if pd.isna(row_dates.loc[idx]) or donors.empty:
                continue

            current_quality = row_quality.loc[idx]
            better = donors
            if pd.notna(current_quality):
                better = donors.loc[donors["_quality"].lt(current_quality)]
            if better.empty:
                continue

            distance = (better["_date"] - row_dates.loc[idx]).abs()
            nearest = better.loc[distance.eq(distance.min())]

            # Equally close but different coordinates: leave the row unchanged.
            if nearest["byg404Koordinat"].nunique() != 1:
                continue

            best = nearest.loc[nearest["_quality"].idxmin()]
            for column in [
                "byg404Koordinat",
                "byg134KvalitetAfKoordinatsæt",
                "byg406Koordinatsystem",
            ]:
                out.loc[idx, column] = best[column]

    # 2. Construction year: latest dated row with a nonmissing year.
    # A date tie uses the first such row in the original CSV.
    if overwrite_construction_year:
        usable = dates.notna() & history["byg026Opførelsesår"].notna()
        if usable.any():
            latest_idx = dates.loc[usable].idxmax()
            out["byg026Opførelsesår"] = history.loc[
                latest_idx, "byg026Opførelsesår"
            ]

    # 3. Replace only retired use codes.
    # The latest date must have one unique, nonmissing, non-retired code.
    if overwrite_main_use and dates.notna().any():
        latest = history.loc[dates.eq(dates.max()), "byg021BygningensAnvendelse"]
        latest_codes = pd.to_numeric(latest, errors="coerce").dropna().unique()

        if len(latest_codes) == 1:
            new_code = latest_codes[0]
            if new_code not in UDFASEDE_ANVENDELSESKODER:
                old_codes = pd.to_numeric(
                    rows["byg021BygningensAnvendelse"], errors="coerce"
                )
                replace = old_codes.isin(UDFASEDE_ANVENDELSESKODER)
                out.loc[replace, "byg021BygningensAnvendelse"] = str(int(new_code))

    return out



def _append_partitions(chunk, folder, prefix, level, counts):
    """Send each building to the same temporary CSV file."""
    buckets = (chunk["_bbr_hash"] // (64 ** level)) % 64
    for bucket, part in chunk.groupby(buckets, sort=False):
        path = folder / f"{prefix}_{int(bucket):02d}.csv"
        part.to_csv(path, mode="a", header=not path.exists(), index=False)
        counts[path] = counts.get(path, 0) + len(part)


def filtrer_bbr_csv_OPFØRT_a_AktuelleHistoriskeData(
    input_folder,
    input_filename,
    output_folder,
    output_filename,
    status=6,
    cols=None,
    overwrite_coordinate=True,
    overwrite_construction_year=True,
    overwrite_main_use=True,
    chunksize=100_000,
):
    """Filter and correct using pandas and temporary CSV files.

    Original retained rows supply all corrections. Missing building IDs are
    retained without correction. Output keeps the input row/column order.

    Temporary files are split until they contain at most chunksize rows,
    except when one building's history alone exceeds that limit.
    """
    if not isinstance(chunksize, int) or isinstance(chunksize, bool) or chunksize < 1:
        raise ValueError("chunksize must be a positive integer.")

    input_path = Path(input_folder) / input_filename
    output_path = Path(output_folder) / output_filename
    if input_path.resolve() == output_path.resolve():
        raise ValueError("Input and output must be different files.")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # 1. Choose output columns and the columns needed for corrections.
    available = pd.read_csv(input_path, nrows=0).columns.tolist()
    output_columns = available if cols is None else list(cols)
    corrections = any([
        overwrite_coordinate, overwrite_construction_year, overwrite_main_use
    ])
    keys = ["id_namespace", "id_lokalId"]
    required = ["registreringTil", "status"]
    if corrections:
        required += keys + ["virkningFra"]
    if overwrite_coordinate:
        required += [
            "byg134KvalitetAfKoordinatsæt",
            "byg404Koordinat",
            "byg406Koordinatsystem",
        ]
    if overwrite_construction_year:
        required += ["byg026Opførelsesår"]
    if overwrite_main_use:
        required += ["byg021BygningensAnvendelse"]

    needed = list(dict.fromkeys(output_columns + required))
    missing = sorted(set(needed) - set(available))
    if missing:
        raise ValueError(f"Missing CSV columns: {missing}")
    if len(output_columns) != len(set(output_columns)):
        raise ValueError("cols contains duplicate column names.")
    if {"_bbr_row_order", "_bbr_hash"} & set(available):
        raise ValueError("Input contains reserved internal column names.")

    part_dtypes = {column: "string" for column in needed}
    part_dtypes.update(_bbr_row_order="int64", _bbr_hash="uint64")
    rows_read = 0
    rows_written = 0

    with TemporaryDirectory(prefix="bbr_chunks_", dir=output_path.parent) as temp_folder:
        folder = Path(temp_folder)
        temporary_output = folder / "result.csv"
        pd.DataFrame(columns=output_columns).to_csv(temporary_output, index=False)

        # 2. Read/filter in chunks. Put each building's retained history
        #    into one temporary CSV, including rows from later input chunks.
        counts = {}
        with pd.read_csv(
            input_path, usecols=needed, dtype="string", chunksize=chunksize,
        ) as reader:
            for chunk in tqdm(reader, desc="Reading and filtering", unit="chunk"):
                rows_read += len(chunk)
                keep = (
                    chunk["registreringTil"].isna()
                    & pd.to_numeric(chunk["status"], errors="coerce").eq(status)
                )
                filtered = chunk.loc[keep].copy()
                rows_written += len(filtered)
                if filtered.empty:
                    continue

                filtered["_bbr_row_order"] = filtered.index.astype("int64")
                if corrections:
                    filtered["_bbr_hash"] = pd.util.hash_pandas_object(
                        filtered[keys], index=False
                    )
                    # Missing IDs cannot define a building; distribute those rows individually.
                    missing_id = filtered[keys].isna().any(axis=1)
                    filtered.loc[missing_id, "_bbr_hash"] = pd.util.hash_pandas_object(
                        filtered.loc[missing_id, ["_bbr_row_order"]], index=False
                    )
                    _append_partitions(filtered, folder, "part", 0, counts)
                else:
                    for column in [
                        "byg406Koordinatsystem", "registreringFra", "virkningFra", "virkningTil"
                    ]:
                        if column in filtered:
                            filtered[column] = filtered[column].str.replace("Z", "", regex=False)
                    filtered.to_csv(
                        temporary_output, mode="a", header=False,
                        index=False, columns=output_columns,
                    )

        # 3. Correct one small temporary file at a time.
        #    If a file is too large, split it into smaller CSV files first.
        pending = [(path, count, 1) for path, count in counts.items()]
        with tqdm(total=rows_written, desc="Correcting buildings", unit="row") as progress:
            if not corrections:
                progress.update(rows_written)
            while pending:
                path, count, level = pending.pop()
                if count > chunksize:
                    if level > 10:
                        raise ValueError("Could not split a temporary file into small groups.")

                    children = {}
                    first_key = None
                    one_building = True
                    with pd.read_csv(
                        path, dtype=part_dtypes, chunksize=chunksize,
                    ) as reader:
                        for chunk in reader:
                            if first_key is None:
                                first_key = tuple(chunk[keys].iloc[0])
                            if one_building:
                                one_building = bool(
                                    chunk[keys].notna().all().all()
                                    and chunk[keys[0]].eq(first_key[0]).all()
                                    and chunk[keys[1]].eq(first_key[1]).all()
                                )
                            _append_partitions(
                                chunk, folder, path.stem, level, children
                            )
                    if not one_building:
                        pending.extend(
                            (child, size, level + 1) for child, size in children.items()
                        )
                        path.unlink()
                        continue

                    # A single building cannot be split without losing its history.
                    for child in children:
                        child.unlink()

                part = pd.read_csv(path, dtype=part_dtypes)
                part = part.sort_values("_bbr_row_order", kind="stable")
                for (namespace, building_id), rows in part.groupby(
                    keys, sort=False, dropna=False
                ):
                    if pd.isna(namespace) or pd.isna(building_id):
                        continue
                    corrected = _ret_bygning(
                        rows, rows, overwrite_coordinate,
                        overwrite_construction_year, overwrite_main_use,
                    )
                    part.loc[rows.index, corrected.columns] = corrected

                for column in [
                    "byg406Koordinatsystem", "registreringFra", "virkningFra", "virkningTil"
                ]:
                    if column in part:
                        part[column] = part[column].str.replace("Z", "", regex=False)

                # Send corrected rows back to their original input chunk.
                original_chunks = part["_bbr_row_order"] // chunksize
                for chunk_number, rows in part.groupby(original_chunks, sort=False):
                    target = folder / f"output_{int(chunk_number):09d}.csv"
                    rows.to_csv(
                        target, mode="a", header=not target.exists(), index=False,
                        columns=output_columns + ["_bbr_row_order"],
                    )
                path.unlink()
                progress.update(len(part))

        # 4. Restore original row order and write the final CSV in chunks.
        for path in tqdm(sorted(folder.glob("output_*.csv")), desc="Writing output", unit="chunk"):
            part = pd.read_csv(
                path,
                dtype={**{column: "string" for column in output_columns},
                       "_bbr_row_order": "int64"},
            ).sort_values("_bbr_row_order", kind="stable")
            part.to_csv(
                temporary_output, mode="a", header=False, index=False,
                columns=output_columns,
            )
            path.unlink()

        # Replace an existing output only after the entire run succeeds.
        temporary_output.replace(output_path)

    print(f"Read: {rows_read:,}; saved: {rows_written:,}; removed: {rows_read - rows_written:,}")
    print(f"Saved to: {output_path}")
    return output_path


# Keep your folders, filenames and cols_to_extract unchanged.
# Call with chunksize=100_000; reduce it further if needed.


# Keep your settings and cols_to_extract unchanged. Add chunksize to the call:
input_folder = "D:/tung_databehandling/BBR/raw" # data_folder_path_output = "P:/cmo/TFP&Tæthed/BBR/data"
input_filename = "BBR_V3_Bygning_TotalDownload_csv_Bitemporal_653.csv"
output_folder = "D:/tung_databehandling/BBR/raw"
output_filename = "BBR_V3_Bygning_TotalDownload_csv_Bitemporal_653_registreringTil_isna_Status6_fixedKoordinater_fixedOpførelsesår_fixedAnvendelseskoder_selected_cols_chunks.csv"

cols_to_extract = ['id_namespace', # identifikation af et namespace inden for hvilket lokalId er unik
 'id_lokalId', # identifikation af objektet
 'kommunekode', # se kodeliste: https://teknik.bbr.dk/kodelister/0/1/0/Kommunekode
 'registreringFra',
 'registreringTil',
 'virkningFra', # tidspunktet hvor virkningen af den pågældende version af bygværkselementet er startet
 'virkningTil', # alle uden værdi (NaN)
 'status', # kode for bygværkselementets status i den pågældende version, dvs. elementets tilstand i den samlede livscyklus
 'byg021BygningensAnvendelse', # se kodeliste: https://teknik.bbr.dk/kodelister/0/1/0/BygAnvendelse
 'byg026Opførelsesår', # angiver bygningens opførselsår
 'byg027OmTilbygningsår', # Bygningens seneste om- eller tilbygningsår. Angives i formatet åååå
 'byg038SamletBygningsareal', # Det samlede bygningsareal er summen af etagernes arealer, men ekskl. kælderareal og areal af tagetage. Arealerne måles til ydersiden af ydervæggene (bruttoetageareal). Værdisæt 0 – 300.000 (kan være negativ ved byggesagsdata)
 'byg039BygningensSamledeBoligAreal', # Areal, der anvendes til boligformål, !!!herunder også udnyttet areal af tagetage og lovlig beboelse i kælderetage!!!. Arealet måles til ydersiden af ydervægge (bruttoetageareal). Arealet er inkl. evt. andel af adgangsarealer. Værdisæt 0 – 300.000 (kan være negativ ved byggesagsdata)
 'byg040BygningensSamledeErhvervsAreal', # Areal, der udelukkende anvendes til erhverv (ikke-boligformål), herunder også udnyttet areal af tagetage og kælderetage. Arealet måles til ydersiden af ydervægge (bruttoetageareal). Arealet er inkl. evt. andel af adgangsarealer. Værdisæt 0 – 300.000 (kan være negativ ved byggesagsdata)
 'byg041BebyggetAreal', # Arealet registreres som stueetagens areal målt til ydersiden af ydervæggene. For bygninger, hvor etagerne arealmæssigt afviger fra hinanden, angives arealet af den største etage. Kælderetagen registreres særskilt i felt ETA.20 i ''kælder''. Arealet af åbne, overdækkede terrasser, åbne altaner og åbne indgangspartier medregnes ikke. Såfremt sådanne arealer lukkes, skal arealet medregnes. Arealer af porte, luftsluser o. lign. medtages til den linie, som bygningens ydervægge iøvrigt angiver. Arealet af garage, carport eller udhus, der udgør en del af bygningen (indenfor bygningens ydervægge og under samme tag) medregnes ligeledes. Værdisæt 0 – 300.000 (kan være negativ ved byggesagsdata)
 'byg054AntalEtager',
 #'byg070Fredning', # personligt nysgerrig hvor mange der er og hvad deres karakteristika er:) - ikke relevant for agglomerationsprojektet
 'byg134KvalitetAfKoordinatsæt', # kodeliste der angiver kvaliteten af geometrioplysninger (1: Sikker, 2(udfases): næsten sikker, 3: usikker)
 'byg404Koordinat', # Angiver bygningens geografiske repræsentation i form af et punkt
 'byg406Koordinatsystem'] # burde bruge 5 = UTM 32 Euref89 (= epsg:25832)

filtrer_bbr_csv_OPFØRT_a_AktuelleHistoriskeData(
    input_folder, input_filename, output_folder, output_filename,
    status=6,
    cols=cols_to_extract,
    overwrite_coordinate=True,
    overwrite_construction_year=True,
    overwrite_main_use=True,
    chunksize=2_000_000,
)

Reading and filtering: 0chunk [00:00, ?chunk/s]

In [ ]:
def filtrer_bbr_csv_OPFØRT_a_AktuelleHistoriskeData(
    input_folder,
    input_filename,
    output_folder,
    output_filename,
    status=6,
    cols=None,
    overwrite_coordinate=True,
    overwrite_construction_year=True,
    overwrite_main_use=True,
):
    keys = ["id_namespace", "id_lokalId"]
    anvendelse = "byg021BygningensAnvendelse"

    # dette er koderne som ikke længere er aktuelle i BBR registret. Disse overskrives med hver bygning id's nyeste registrering.
    udfasede_koder = [
        130, 210, 220, 230, 290, 310, 320, 330,
        390, 410, 420, 430, 440, 490, 520, 530,
    ]

    # 1. Indlæs data
    required = keys + [
        "status",
        "registreringTil",
        "virkningFra",
        "byg404Koordinat",
        "byg134KvalitetAfKoordinatsæt",
        "byg026Opførelsesår",
    ]

    if overwrite_main_use:
        required.append(anvendelse)

    usecols = (
        None
        if cols is None
        else list(dict.fromkeys(list(cols) + required))
    )

    df = pd.read_csv(
        f"{input_folder}/{input_filename}",
        usecols=usecols,
    )

    # 2. Rens datoer og behold opførte, aktuelt registrerede rækker
    for col in [
        "registreringFra", "registreringTil",
        "virkningFra", "virkningTil",
    ]:
        if col in df:
            df[col] = (
                df[col].astype("string")
                .str.replace("Z", "", regex=False)
            )

    df["status"] = pd.to_numeric(df["status"], errors="coerce")

    df = df[
        df["registreringTil"].isna()
        & df["status"].eq(status)
    ].copy()

    # 3. Opret hjælpekolonner
    df["_dato"] = pd.to_datetime(df["virkningFra"], errors="coerce")

    df["_kvalitet"] = pd.to_numeric(
        df["byg134KvalitetAfKoordinatsæt"],
        errors="coerce",
    )

    if overwrite_main_use:
        df["_kode"] = pd.to_numeric(df[anvendelse], errors="coerce")

    # Behandl én bygning ad gangen
    for _, bygning in df.groupby(keys, sort=False, dropna=False):

        # Gem originalerne: ændringer skrives kun til df
        bygning = bygning.copy()

        # 4. Erstat koordinater med nærmeste af strengt bedre kvalitet
        if overwrite_coordinate:
            kandidater = bygning.dropna(
                subset=["byg404Koordinat", "_dato", "_kvalitet"]
            )

            for i, række in bygning.iterrows():
                if pd.isna(række["_dato"]):
                    continue

                if pd.isna(række["_kvalitet"]):
                    bedre = kandidater
                else:
                    bedre = kandidater[
                        kandidater["_kvalitet"] < række["_kvalitet"]
                    ]

                if bedre.empty:
                    continue

                afstand = (bedre["_dato"] - række["_dato"]).abs()
                nærmeste = bedre[afstand.eq(afstand.min())]

                # Forskellige koordinater ved samme afstand: ændr intet
                if nærmeste["byg404Koordinat"].nunique() != 1:
                    continue

                valgt = nærmeste.sort_values("_kvalitet").iloc[0]

                df.at[i, "byg404Koordinat"] = valgt["byg404Koordinat"]
                df.at[i, "byg134KvalitetAfKoordinatsæt"] = (
                    valgt["byg134KvalitetAfKoordinatsæt"]
                )

        # Sortér registreringer med gyldig dato fra ældst til nyest
        daterede = (
            bygning.dropna(subset=["_dato"])
            .sort_values("_dato", kind="stable")
        )

        # 5. Brug seneste gyldige opførelsesår i alle perioder
        if overwrite_construction_year:
            gyldige_år = daterede.dropna(
                subset=["byg026Opførelsesår"]
            )

            if not gyldige_år.empty:
                seneste_år = gyldige_år.iloc[-1]["byg026Opførelsesår"]
                df.loc[bygning.index, "byg026Opførelsesår"] = seneste_år

        # 6. Erstat KUN udfasede anvendelseskoder med seneste kode
        if overwrite_main_use and not daterede.empty:
            udfaset = bygning["_kode"].isin(udfasede_koder)

            if not udfaset.any():
                continue

            seneste_dato = daterede.iloc[-1]["_dato"]
            seneste = daterede[daterede["_dato"].eq(seneste_dato)]
            koder = seneste["_kode"].unique()

            # Seneste kode skal være entydig, ikke mangle og ikke være udfaset
            if (
                len(koder) == 1
                and pd.notna(koder[0])
                and koder[0] not in udfasede_koder
            ):
                indeks = bygning.index[udfaset]
                df.loc[indeks, anvendelse] = seneste.iloc[0][anvendelse]

    # 7. Fjern hjælpekolonner og rens koordinatsystem
    df = df.drop(columns=["_dato", "_kvalitet", "_kode"], errors="ignore")

    if "byg406Koordinatsystem" in df:
        df["byg406Koordinatsystem"] = (
            df["byg406Koordinatsystem"]
            .astype("string")
            .str.replace("Z", "", regex=False)
        )

    # 8. Behold ønskede outputkolonner og gem
    if cols is not None:
        df = df[[col for col in cols if col in df]]

    df.to_csv(
        f"{output_folder}/{output_filename}",
        index=False,
    )

    print(f"Gemte rækker: {len(df):,}")
    print("Done!")


# Keep your settings and cols_to_extract unchanged. Add chunksize to the call:
input_folder = "D:/tung_databehandling/BBR/raw" # data_folder_path_output = "P:/cmo/TFP&Tæthed/BBR/data"
input_filename = "BBR_V3_Bygning_TotalDownload_csv_Bitemporal_653.csv"
output_folder = "D:/tung_databehandling/BBR/raw"
output_filename = "BBR_V3_Bygning_TotalDownload_csv_Bitemporal_653_registreringTil_isna_Status6_fixedKoordinater_fixedOpførelsesår_fixedAnvendelseskoder_selected_cols.csv"

cols_to_extract = ['id_namespace', # identifikation af et namespace inden for hvilket lokalId er unik
 'id_lokalId', # identifikation af objektet
 'kommunekode', # se kodeliste: https://teknik.bbr.dk/kodelister/0/1/0/Kommunekode
 'registreringFra',
 'registreringTil',
 'virkningFra', # tidspunktet hvor virkningen af den pågældende version af bygværkselementet er startet
 'virkningTil', # alle uden værdi (NaN)
 'status', # kode for bygværkselementets status i den pågældende version, dvs. elementets tilstand i den samlede livscyklus
 'byg021BygningensAnvendelse', # se kodeliste: https://teknik.bbr.dk/kodelister/0/1/0/BygAnvendelse
 'byg026Opførelsesår', # angiver bygningens opførselsår
 'byg027OmTilbygningsår', # Bygningens seneste om- eller tilbygningsår. Angives i formatet åååå
 'byg038SamletBygningsareal', # Det samlede bygningsareal er summen af etagernes arealer, men ekskl. kælderareal og areal af tagetage. Arealerne måles til ydersiden af ydervæggene (bruttoetageareal). Værdisæt 0 – 300.000 (kan være negativ ved byggesagsdata)
 'byg039BygningensSamledeBoligAreal', # Areal, der anvendes til boligformål, !!!herunder også udnyttet areal af tagetage og lovlig beboelse i kælderetage!!!. Arealet måles til ydersiden af ydervægge (bruttoetageareal). Arealet er inkl. evt. andel af adgangsarealer. Værdisæt 0 – 300.000 (kan være negativ ved byggesagsdata)
 'byg040BygningensSamledeErhvervsAreal', # Areal, der udelukkende anvendes til erhverv (ikke-boligformål), herunder også udnyttet areal af tagetage og kælderetage. Arealet måles til ydersiden af ydervægge (bruttoetageareal). Arealet er inkl. evt. andel af adgangsarealer. Værdisæt 0 – 300.000 (kan være negativ ved byggesagsdata)
 'byg041BebyggetAreal', # Arealet registreres som stueetagens areal målt til ydersiden af ydervæggene. For bygninger, hvor etagerne arealmæssigt afviger fra hinanden, angives arealet af den største etage. Kælderetagen registreres særskilt i felt ETA.20 i ''kælder''. Arealet af åbne, overdækkede terrasser, åbne altaner og åbne indgangspartier medregnes ikke. Såfremt sådanne arealer lukkes, skal arealet medregnes. Arealer af porte, luftsluser o. lign. medtages til den linie, som bygningens ydervægge iøvrigt angiver. Arealet af garage, carport eller udhus, der udgør en del af bygningen (indenfor bygningens ydervægge og under samme tag) medregnes ligeledes. Værdisæt 0 – 300.000 (kan være negativ ved byggesagsdata)
 'byg054AntalEtager',
 #'byg070Fredning', # personligt nysgerrig hvor mange der er og hvad deres karakteristika er:) - ikke relevant for agglomerationsprojektet
 'byg134KvalitetAfKoordinatsæt', # kodeliste der angiver kvaliteten af geometrioplysninger (1: Sikker, 2(udfases): næsten sikker, 3: usikker)
 'byg404Koordinat', # Angiver bygningens geografiske repræsentation i form af et punkt
 'byg406Koordinatsystem'] # burde bruge 5 = UTM 32 Euref89 (= epsg:25832)


filtrer_bbr_csv_OPFØRT_a_AktuelleHistoriskeData(
    input_folder, input_filename, output_folder, output_filename,
    status=6,
    cols=cols_to_extract,
    overwrite_coordinate=True,
    overwrite_construction_year=True,
    overwrite_main_use=True,
)

# skal slettes (dette er lokalebasen_merge_dar_brr.ipynb koden)

In [ ]:
# %% Lokalebasen -> classification/facilities/CPI -> DAR -> BBR -> GIS
# Dependencies: pandas, numpy, openpyxl, httpx, pyproj.
# CPI is attached as cpi (2025 = 100); rents remain in their original units.
# BBR reference date: 1 July of each observation year.
# Enhed route permits before/after-period fallbacks; direct route requires a valid period.
# The classification join is inner: leases without classification are excluded.

from __future__ import annotations

from pathlib import Path
import re
import numpy as np
import pandas as pd
from difflib import SequenceMatcher

# =============================================================================
# SETTINGS
# =============================================================================

# Output routes:
#   adresse_id -> BBR Enhed -> BBR Bygning -> GIS
#   adgangsadresseid -> BBR Bygning -> GIS
#   DAR match -> GIS
#
# GISX/GISY are always returned in EPSG:25832.

RAW_LOKALEBASEN_FOLDER = Path(
    r"D:\tung_databehandling\lokalebasen\data\raw"
)
PROCESSED_FOLDER = Path(
    r"D:\tung_databehandling\lokalebasen\data\processed"
)
CLASSIFICATION_PATH = (
    RAW_LOKALEBASEN_FOLDER / "lejemalsklassifikation.csv"
)
FACILITY_PATH = (
    RAW_LOKALEBASEN_FOLDER / "lejemalsklassifikation_facility_dummies.csv"
)
CPI_PATH = (
    RAW_LOKALEBASEN_FOLDER / "PRIS02_DST_2025eq100.xlsx"
)

# Klassifikationskolonnerne kommer fra den oprindelige klassifikationsfil.
CLASSIFICATION_VALUE_COLUMNS = [
    "andel_kontor",
    "andel_butik",
    "andel_lager",
    "andel_andet",
    "hovedkategori",
    "hovedkat_andel",
    "vurdering_mulig",
    "sikkerhed",
    "andel_kilde",
]

# Manuel og gennemsigtig liste: kun disse facility-dummies føres videre.
RELEVANT_FACILITY_COLUMNS = [
    "facility_mangler",
    "facility_alarmsystem",
    "facility_arkiv_og_lagerplads",
    "facility_auditorium",
    "facility_bad_og_omklaedning",
    "facility_elevator",
    "facility_faelles_kantine",
    "facility_faelles_loungeomraade",
    "facility_faelles_moedelokaler",
    "facility_gaardanlaeg",
    "facility_handikapvenligt",
    "facility_it_support",
    "facility_kaeledyrsegnet",
    "facility_klimaanlaeg",
    "facility_koekken",
    "facility_konferencerum",
    "facility_kopieringsfaciliteter",
    "facility_parkeringspladser",
    "facility_reception",
    "facility_traeningscenter",
    "facility_videoovervaagning",
]

BBR_ENHED_PATH = Path(
    r"D:\tung_databehandling\BBR\raw"
    r"\BBR_V3_Enhed_TotalDownload_csv_Bitemporal_653_"
    r"registreringTil_isna_Status6.csv"
)
BBR_BYGNING_PATH = Path(
    r"D:\tung_databehandling\BBR\raw"
    r"\BBR_V3_Bygning_TotalDownload_csv_Bitemporal_653_"
    r"registreringTil_isna_Status6_latestKoordinat_latestOpførelsesår.csv"
)

# This is the ONLY cache read by this program.
# Complete validated rows from this cache can be reused.
# Negative legacy results are retried through Adressevaelger.
DAR_CACHE_PATH = PROCESSED_FOLDER / "dar_lookup_fresh2.csv"

OUT_FULL_PATH = (
    PROCESSED_FOLDER
    / "lokalebasen_dar_bbr_pris02_enhed_bygning_full_strict.csv"
)
OUT_GIS_PATH = (
    PROCESSED_FOLDER
    / "lokalebasen_dar_bbr_pris02_enhed_bygning_GIS_slim_strict.csv"
)

SAVE_EVERY = 500
BBR_CHUNK_SIZE = 500_000
AUTOCOMPLETE_RESULTS = 5
DAR_LOOKUP_VERSION = "adressevaelger_strict_v2"
DAR_BASE_URL = "https://adressevaelger.dk"
DAR_TOKEN = "adressevaelger123"  # Public token recommended by KDS.

# False resumes safely from DAR_CACHE_PATH after an interrupted run.
# True ignores that one cache and starts all DAR lookups again.
REBUILD_DAR_CACHE = False


# =============================================================================
# GENERIC HELPERS
# =============================================================================

def is_missing(value) -> bool:
    if value is None:
        return True
    try:
        missing = pd.isna(value)
    except (TypeError, ValueError):
        return False
    return bool(missing) if isinstance(missing, (bool, np.bool_)) else False


def clean_address(value):
    if is_missing(value):
        return np.nan
    text = re.sub(r"\s+", " ", str(value).strip())
    text = re.sub(r"\s*,\s*", ", ", text).strip(" ,")
    return text if text else np.nan


def normalize_id(series: pd.Series) -> pd.Series:
    return series.astype("string").str.lower().str.strip()


def normalize_location_id(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0+$", "", regex=True)
    )


def normalize_kommunekode(series: pd.Series) -> pd.Series:
    text = (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0+$", "", regex=True)
    )
    return text.str.extract(r"^(\d{1,4})$", expand=False).str.zfill(4)


def as_bool(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes", "ja"})
    )


def similarity(left, right):
    if is_missing(left) or is_missing(right):
        return None
    return round(
        100 * SequenceMatcher(
            None,
            str(left).casefold(),
            str(right).casefold(),
        ).ratio(),
        2,
    )


def parse_point_to_xy(value):
    if is_missing(value):
        return pd.Series({"GISX_bbr": np.nan, "GISY_bbr": np.nan})

    match = re.search(
        r"POINT\s*(?:Z|M|ZM)?\s*\(\s*([0-9.\-]+)\s+([0-9.\-]+)",
        str(value),
        flags=re.IGNORECASE,
    )
    if not match:
        return pd.Series({"GISX_bbr": np.nan, "GISY_bbr": np.nan})

    return pd.Series({
        "GISX_bbr": float(match.group(1)),
        "GISY_bbr": float(match.group(2)),
    })


def load_and_prepare_lookup(path, value_columns, label):
    """One classification/facility row per year and location."""
    frame = pd.read_csv(
        path, usecols=["source_year", "Location ID"] + value_columns,
        dtype={"Location ID": "string"}, low_memory=False,
    )
    frame = frame.rename(columns={"source_year": "year", "Location ID": "location_id"})
    frame["year"] = pd.to_numeric(frame["year"], errors="raise").astype("Int64")
    frame["location_id"] = normalize_location_id(frame["location_id"])

    if frame["year"].isna().any() or frame["location_id"].fillna("").eq("").any():
        raise ValueError(f"{label}: missing year/location_id.")
    if frame.duplicated(["year", "location_id"]).any():
        raise ValueError(f"{label}: duplicate year/location_id.")
    return frame


def load_cpi():
    """Read the exact Excel layout shown in the question (2025 = 100)."""
    pris02 = pd.read_excel(CPI_PATH)
    cpi = pd.DataFrame({
        "year": pd.to_numeric(pris02.iloc[1, 1:], errors="raise").astype("Int64").to_numpy(),
        "cpi": pd.to_numeric(pris02.iloc[2, 1:], errors="raise").to_numpy(),
    })
    if cpi["year"].isna().any() or cpi["year"].duplicated().any():
        raise ValueError("CPI years must be present and unique.")
    if cpi["cpi"].isna().any() or not np.isfinite(cpi["cpi"]).all() or cpi["cpi"].le(0).any():
        raise ValueError("CPI values must be finite and positive.")
    return cpi


def add_cpi(df):
    """Keep every input row and attach its annual CPI."""
    out = df.merge(load_cpi(), on="year", how="left", validate="many_to_one", sort=False)
    missing_years = out.loc[out["cpi"].isna(), "year"].drop_duplicates().tolist()
    if missing_years:
        raise ValueError(f"CPI is missing for years: {missing_years}")
    return out

def load_lokalebasen() -> pd.DataFrame:
    files = sorted(
        path
        for path in RAW_LOKALEBASEN_FOLDER.glob("*.csv")
        if re.search(r"\d{4}", path.name)
        and path not in {CLASSIFICATION_PATH, FACILITY_PATH, CPI_PATH}
        and not path.name.upper().startswith("PRIS02")
    )
    if not files:
        raise FileNotFoundError(
            "No yearly Lokalebasen CSV files found in "
            f"{RAW_LOKALEBASEN_FOLDER}"
        )

    parts = []
    for path in files:
        year_match = re.search(r"\d{4}", path.name)
        part = pd.read_csv(path, low_memory=False)
        part["year"] = int(year_match.group())
        parts.append(part)

    raw = pd.concat(parts, ignore_index=True)

    # Density files use this position in the concatenated raw files as `id`.
    # It must be created before the inner classification join removes rows.
    raw.insert(0, "density_id", raw.index.astype("int64"))

    # The same measure changed name after 2009. Combine it before renaming so
    # `total_shows` is populated for every year, not only the old file.
    if "Total count per year" in raw.columns:
        if "Total shows" in raw.columns:
            raw["Total shows"] = raw["Total shows"].combine_first(
                raw["Total count per year"]
            )
        else:
            raw["Total shows"] = raw["Total count per year"]

    raw = raw.rename(columns={
        "Location ID": "location_id",
        "Address": "address",
        "Yearly rent per m2 amount from": "rent_from",
        "Yearly rent per m2 amount to": "rent_to",
        "Title": "title",
        "Description": "description",
        "Facilities": "facilities",
        "Contact details shown": "contact_details_shown",
        "Total shows": "total_shows",
        "Energy rating": "energy_rating",
    })
    keep = [
        "density_id",
        "location_id",
        "year",
        "address",
        "rent_from",
        "rent_to",
        "title",
        "description",
        "facilities",
        "contact_details_shown",
        "total_shows",
        "energy_rating",
    ]
    missing = set(keep) - set(raw.columns)
    if missing:
        raise ValueError(f"Lokalebasen is missing columns: {sorted(missing)}")

    raw = raw[keep].copy()
    raw["address_clean"] = raw["address"].map(clean_address)

    classification = load_and_prepare_lookup(
        CLASSIFICATION_PATH,
        CLASSIFICATION_VALUE_COLUMNS,
        "Classification",
    )
    facilities = load_and_prepare_lookup(
        FACILITY_PATH,
        RELEVANT_FACILITY_COLUMNS,
        "Facility",
    )

    # Both source files must describe exactly the same leases.  An outer
    # one-to-one merge makes any future source drift explicit instead of
    # silently dropping classifications or facilities.
    enrichment = classification.merge(
        facilities,
        on=["year", "location_id"],
        how="outer",
        validate="one_to_one",
        indicator="_facility_merge",
    )
    if enrichment["_facility_merge"].ne("both").any():
        raise ValueError("Classification and facility year/location_id keys differ.")
    enrichment = enrichment.drop(columns=["_facility_merge"])

    raw["year"] = pd.to_numeric(raw["year"], errors="raise").astype("Int64")
    raw["location_id"] = normalize_location_id(raw["location_id"])

    before = len(raw)
    out = raw.merge(
        enrichment,
        on=["year", "location_id"],
        how="inner",
        validate="many_to_one",
    )
    out["id"] = out["density_id"]
    out = add_cpi(out)

    print(f"Loaded {len(out):,} leases; excluded {before - len(out):,} without classification.")

    return out


# =============================================================================
# ADDRESS PARSING
# =============================================================================

def normalize_postnr(value):
    if is_missing(value):
        return None
    matches = re.findall(r"\b\d{4}\b", str(value))
    if matches:
        return matches[-1]
    text = str(value).strip()
    return text.zfill(4) if re.fullmatch(r"\d{1,4}", text) else None


def extract_postcode_city(address):
    if is_missing(address):
        return None
    matches = list(re.finditer(r"\b\d{4}\b", str(address)))
    if not matches:
        return None
    return str(address)[matches[-1].start():].strip(" ,")


def normalize_street(value):
    if is_missing(value):
        return None
    text = str(value)
    replacements = {
        r"\bsdr\.?(?=\s|$)": "søndre",
        r"\bndr\.?(?=\s|$)": "nordre",
        r"\bgl\.?(?=\s|$)": "gammel",
    }
    for pattern, replacement in replacements.items():
        text = re.sub(
            pattern,
            replacement,
            text,
            flags=re.IGNORECASE,
        )
    text = text.casefold().replace("å", "aa")
    text = text.replace("æ", "ae").replace("ø", "oe")
    text = re.sub(r"[^a-z0-9]", "", text)
    return text or None


def normalize_husnr(value):
    if is_missing(value):
        return None
    text = re.sub(r"\s+", "", str(value).casefold()).replace(".", "")
    match = re.search(r"(\d+[a-zæøå]?)", text)
    return match.group(1) if match else None


def first_address_component(address):
    if is_missing(address):
        return None
    text = str(clean_address(address))
    postcode_matches = list(re.finditer(r"\b\d{4}\b", text))
    before_postcode = (
        text[:postcode_matches[-1].start()].strip(" ,")
        if postcode_matches
        else text
    )
    parts = [
        str(clean_address(part))
        for part in before_postcode.split(",")
        if not is_missing(clean_address(part))
    ]
    if not parts:
        return None

    # "A. P. Møllers Allé, 43, 2791 Dragør"
    standalone_house = (
        r"\d+[A-Za-zÆØÅæøå]?"
        r"(?:\s*-\s*(?:\d+[A-Za-zÆØÅæøå]?|[A-Za-zÆØÅæøå]))?"
    )
    if (
        len(parts) >= 2
        and not re.search(r"\d", parts[0])
        and re.fullmatch(standalone_house, parts[1])
    ):
        return f"{parts[0]} {parts[1]}"
    return parts[0]


def remove_trailing_unit_text(value):
    """Remove only complete trailing floor/unit tokens, never street prefixes."""
    if is_missing(value):
        return None
    text = str(value).strip()

    text = re.sub(
        r"\s+\b(?:st\.?\s*[\.\-]?\s*(?:th|tv|mf)|"
        r"\d+\.\s*(?:sal|th|tv|mf))\.?\s*$",
        "",
        text,
        flags=re.IGNORECASE,
    )
    unit_word = (
        r"(?:blok|bygning|opgang|afsnit|unit|lokale|lok|lejemål|lejem|"
        r"kontor|lager|showroom|butik|sal|etage|plan|stueplan|stuen|"
        r"stue|st|underetage|kælder|kaelder|kld|kl|parterre|th|tv|mf|"
        r"dør|doer)"
    )
    previous = None
    while previous != text:
        previous = text
        text = re.sub(
            rf"\s+\b{unit_word}\b\.?(?:\s+[A-Za-zÆØÅæøå0-9]+)?\s*$",
            "",
            text,
            flags=re.IGNORECASE,
        )
    return clean_address(text)


def parse_street_house(address):
    primary = remove_trailing_unit_text(first_address_component(address))
    if is_missing(primary):
        return None, []

    primary = re.sub(
        r"(\d+)\s+([A-Za-zÆØÅæøå])\b",
        r"\1\2",
        str(primary),
    )
    match = re.match(
        r"^(?P<street>.*?)"
        r"(?P<house>\d+[A-Za-zÆØÅæøå]?"
        r"(?:\s*-\s*(?:\d+[A-Za-zÆØÅæøå]?|[A-Za-zÆØÅæøå]))?)\s*$",
        primary,
    )
    if not match:
        return None, []

    street = clean_address(match.group("street"))
    house = re.sub(r"\s+", "", match.group("house"))
    if is_missing(street):
        return None, []

    letter_range = re.fullmatch(
        r"(\d+)([A-Za-zÆØÅæøå])-([A-Za-zÆØÅæøå])",
        house,
    )
    numeric_range = re.fullmatch(
        r"(\d+[A-Za-zÆØÅæøå]?)-(\d+[A-Za-zÆØÅæøå]?)",
        house,
    )
    if letter_range:
        base = letter_range.group(1)
        houses = [
            normalize_husnr(base + letter_range.group(2)),
            normalize_husnr(base + letter_range.group(3)),
        ]
    elif numeric_range:
        houses = [
            normalize_husnr(numeric_range.group(1)),
            normalize_husnr(numeric_range.group(2)),
        ]
    else:
        houses = [normalize_husnr(house)]

    return str(street), list(dict.fromkeys(h for h in houses if h))


def primary_queries(address):
    postcode_city = extract_postcode_city(address)
    street, houses = parse_street_house(address)
    if not postcode_city or not street or not houses:
        return []
    return [
        f"{street} {house}, {postcode_city}"
        for house in houses
    ]


# =============================================================================
# ONE DAR LOOKUP, ONE RESULT SCHEMA, ONE CACHE
# =============================================================================

DAR_RESULT_COLUMNS = [
    "address_clean",
    "matched_address",
    "match_level",
    "match_method",
    "method_order",
    "query_used",
    "lookup_status",
    "lookup_error",
    "lookup_version",
    "adresse_id",
    "adgangsadresseid",
    "x",
    "y",
    "kommunekode",
    "postnr",
    "postnrnavn",
    "vejnavn",
    "husnr",
    "match_score",
    "validation_ok",
    "validation_reason",
    "required_postnr",
    "required_vejnavn",
    "required_husnr_values",
    "candidate_postnr_ok",
    "candidate_vejnavn_ok",
    "candidate_husnr_ok",
    "candidates_checked_n",
    "candidates_rejected_n",
    "methods_tried_n",
    "api_failures_n",
    "detail_complete",
]


def _empty_dar_result(
    address,
    *,
    status,
    method,
    reason,
    error=None,
):
    street, houses = parse_street_house(address)
    result = {column: None for column in DAR_RESULT_COLUMNS}
    result.update({
        "address_clean": address,
        "match_method": method,
        "lookup_status": status,
        "lookup_error": error,
        "lookup_version": DAR_LOOKUP_VERSION,
        "validation_ok": False,
        "validation_reason": reason,
        "required_postnr": normalize_postnr(address),
        "required_vejnavn": street,
        "required_husnr_values": ";".join(houses) if houses else None,
        "candidate_postnr_ok": False,
        "candidate_vejnavn_ok": False,
        "candidate_husnr_ok": False,
        "candidates_checked_n": 0,
        "candidates_rejected_n": 0,
        "methods_tried_n": 0,
        "api_failures_n": 0,
        "detail_complete": False,
    })
    return result


def _candidate_dict(data, kind):
    """Map current Adressevaelger detail fields to the existing output schema."""
    if not isinstance(data, dict) or data.get("status") != "ok":
        return None
    address = data.get("adresse") or {}
    house = address.get("husnummer") if kind == "adresse" else data.get("husnummer")
    if not isinstance(house, dict) or not house.get("id_lokalid"):
        return None
    if kind == "adresse" and not address.get("id_lokalid"):
        return None
    point = house.get("adgangspunkt") or {}
    xy = point.get("koordinater") or {}
    post = house.get("postnummer") or {}
    municipality = house.get("navngivenvejkommunedel") or {}
    return {
        "matched_address": address.get("adressebetegnelse") or house.get("adgangsadressebetegnelse"),
        "adresse_id": address.get("id_lokalid") if kind == "adresse" else None,
        "adgangsadresseid": house["id_lokalid"],
        "x": xy.get("x"), "y": xy.get("y"),
        "kommunekode": municipality.get("kommune"),
        "postnr": normalize_postnr(post.get("postnr")),
        "postnrnavn": post.get("navn"),
        "vejnavn": house.get("vejnavn"),
        "husnr": normalize_husnr(house.get("husnummertekst")),
    }

def _validate_candidate(address, candidate):
    required_postnr = normalize_postnr(address)
    required_street, required_houses = parse_street_house(address)

    candidate_postnr = candidate.get("postnr")
    candidate_street = candidate.get("vejnavn")
    candidate_house = candidate.get("husnr")

    # Fall back to parsing the displayed designation when component fields are
    # absent from a specific DAR endpoint.
    if not candidate_street or not candidate_house:
        parsed_street, parsed_houses = parse_street_house(
            candidate.get("matched_address")
        )
        candidate_street = candidate_street or parsed_street
        candidate_house = (
            candidate_house
            or (parsed_houses[0] if parsed_houses else None)
        )
    if not candidate_postnr:
        candidate_postnr = normalize_postnr(
            candidate.get("matched_address")
        )

    postnr_ok = bool(
        required_postnr
        and candidate_postnr
        and required_postnr == candidate_postnr
    )
    street_ok = bool(
        required_street
        and candidate_street
        and normalize_street(required_street)
        == normalize_street(candidate_street)
    )
    house_ok = bool(
        candidate_house
        and candidate_house in set(required_houses)
    )

    if not required_postnr:
        reason = "input_postcode_missing"
    elif not required_street or not required_houses:
        reason = "input_street_or_house_number_missing"
    elif not postnr_ok:
        reason = "postcode_mismatch"
    elif not street_ok:
        reason = "street_mismatch"
    elif not house_ok:
        reason = "house_number_mismatch"
    else:
        reason = "same_postcode_street_and_house_number"

    return {
        "ok": postnr_ok and street_ok and house_ok,
        "reason": reason,
        "required_postnr": required_postnr,
        "required_vejnavn": required_street,
        "required_husnr_values": ";".join(required_houses)
        if required_houses else None,
        "candidate_postnr_ok": postnr_ok,
        "candidate_vejnavn_ok": street_ok,
        "candidate_husnr_ok": house_ok,
    }


def lookup_one_dar(client, address):
    """Full address -> access address -> clean query -> address wash."""
    address = clean_address(address)
    result = _empty_dar_result(
        address, status="unmatched", method="no_dar_match",
        reason="no_candidate_passed_strict_validation",
    )
    street, houses = parse_street_house(address)
    if not normalize_postnr(address) or not street or not houses:
        result["validation_reason"] = "input_street_house_or_postcode_missing"
        return result

    stats = dict(candidates_checked_n=0, candidates_rejected_n=0,
                 methods_tried_n=0, api_failures_n=0)
    errors = []

    def fetch(path, params=None):
        stats["methods_tried_n"] += 1
        try:
            response = client.get(
                DAR_BASE_URL + path,
                params={"token": DAR_TOKEN, **(params or {})}, timeout=20,
            )
            response.raise_for_status()
            data = response.json()
            if not isinstance(data, dict):
                raise ValueError("Unexpected DAR response.")
            if data.get("status") not in {None, "ok"}:
                raise ValueError(data.get("beskrivelse", "DAR request failed."))
            return data
        except Exception as exc:
            stats["api_failures_n"] += 1
            errors.append(f"{type(exc).__name__}: {exc}")
            return {}

    def accept(item, method, order, query):
        kind = item.get("type")
        if kind not in {"adresse", "husnummer"} or not item.get("id"):
            return None
        # Search results have a title; detail calls provide IDs, coordinates and municipality.
        parsed_street, parsed_houses = parse_street_house(item.get("titel"))
        preliminary = {
            "matched_address": item.get("titel"), "vejnavn": parsed_street,
            "husnr": parsed_houses[0] if parsed_houses else None,
            "postnr": normalize_postnr(item.get("titel")),
        }
        stats["candidates_checked_n"] += 1
        if not _validate_candidate(address, preliminary)["ok"]:
            stats["candidates_rejected_n"] += 1
            return None

        endpoint = "adresser" if kind == "adresse" else "husnumre"
        match_level = "adresse" if kind == "adresse" else "adgangsadresse"
        candidate = _candidate_dict(fetch(f"/{endpoint}/{item['id']}"), match_level)
        if candidate is None:
            return None
        validation = _validate_candidate(address, candidate)
        if not validation["ok"]:
            stats["candidates_rejected_n"] += 1
            return None

        complete = all(not is_missing(candidate[k]) for k in ["x", "y", "kommunekode"])
        matched = dict(result)
        matched.update(candidate)
        matched.update(validation)
        matched.update(stats)
        matched.update(
            match_level=match_level, match_method=method, method_order=order,
            query_used=query, validation_ok=True, detail_complete=complete,
            lookup_status="matched" if complete else "error",
            lookup_error=None if complete else "DAR detail incomplete; retry next run.",
            validation_reason=validation["reason"],
            match_score=similarity(address, candidate["matched_address"]),
        )
        return matched

    steps = [("adresser", address, "dar_adresse")]
    for query in dict.fromkeys([address] + primary_queries(address)):
        method = "dar_adgangsadresse_original" if query == address else "dar_adgangsadresse_primary"
        steps.append(("husnumre", query, method))

    for order, (endpoint, query, method) in enumerate(steps, start=1):
        data = fetch(f"/{endpoint}/soeg", {"tekst": query, "maksimum": AUTOCOMPLETE_RESULTS})
        for item in data.get("fund", []):
            if endpoint == "adresser" and item.get("type") != "adresse":
                continue
            matched = accept(item, method, order, query)
            if matched is not None:
                return matched

    for query in dict.fromkeys(primary_queries(address) + [address]):
        washed = fetch("/vask/", {"adresse": query}).get("vaskeresultat") or {}
        item = {"type": "adresse", "id": washed.get("adresse_id_lokalid"),
                "titel": washed.get("adressebetegnelse")}
        matched = accept(item, "dar_datavask_adgangsadresse", len(steps) + 1, query)
        if matched is not None:
            return matched

    result.update(stats)
    if errors:
        result.update(lookup_status="error", match_method="dar_request_error",
                      lookup_error=" | ".join(errors),
                      validation_reason="api_error_retry_next_run")
    return result

def _upsert_cache(cache, new_rows):
    frames = [
        frame
        for frame in [cache, pd.DataFrame(new_rows)]
        if frame is not None and not frame.empty
    ]
    if not frames:
        return pd.DataFrame(columns=DAR_RESULT_COLUMNS)

    out = pd.concat(frames, ignore_index=True)
    for column in DAR_RESULT_COLUMNS:
        if column not in out.columns:
            out[column] = pd.NA
    return (
        out[DAR_RESULT_COLUMNS]
        .drop_duplicates("address_clean", keep="last")
        .reset_index(drop=True)
    )


def lookup_many_dar(addresses) -> pd.DataFrame:
    import httpx

    requested = (
        pd.Series(addresses)
        .dropna()
        .map(clean_address)
        .dropna()
        .drop_duplicates()
        .astype(str)
    )

    if DAR_CACHE_PATH.exists() and not REBUILD_DAR_CACHE:
        cache = pd.read_csv(DAR_CACHE_PATH, dtype=str, low_memory=False)
    else:
        cache = pd.DataFrame(columns=DAR_RESULT_COLUMNS)
    cache = _upsert_cache(cache, [])

    current_version = cache["lookup_version"].eq(DAR_LOOKUP_VERSION)
    terminal_no_match = cache["lookup_status"].eq("unmatched")
    successful = (
        cache["lookup_status"].eq("matched")
        & as_bool(cache["detail_complete"])
        & as_bool(cache["validation_ok"])
        & cache["adgangsadresseid"].notna()
    )
    # Reuse validated complete legacy matches; retry legacy negative results.
    known_version = current_version | cache["lookup_version"].eq("fresh_single_cache_v1")
    reusable = (current_version & terminal_no_match) | (known_version & successful)
    done = set(
        cache.loc[reusable, "address_clean"].dropna().astype(str)
    )
    todo = [address for address in requested if address not in done]

    print("\n--- One-pass DAR lookup ---")
    print("Unique addresses:", f"{len(requested):,}")
    print("Reusable rows in the one cache:", f"{len(done):,}")
    print("Need lookup:", f"{len(todo):,}")
    print("Only cache used:", DAR_CACHE_PATH)
    print(
        "Acceptance: same street -> compatible house number -> same postcode"
    )

    pending = []
    with httpx.Client() as client:
        for number, address in enumerate(todo, start=1):
            pending.append(lookup_one_dar(client, address))
            if number % SAVE_EVERY == 0:
                cache = _upsert_cache(cache, pending)
                cache.to_csv(DAR_CACHE_PATH, index=False)
                pending = []
                print(f"Saved after {number:,} DAR lookups")

    cache = _upsert_cache(cache, pending)
    cache.to_csv(DAR_CACHE_PATH, index=False)

    return cache.loc[
        cache["address_clean"].isin(set(requested))
    ].copy()


# =============================================================================
# BBR INPUT AND TEMPORAL SELECTION
# =============================================================================

ROW_ID_COL = "_lb_row_id"
TARGET_DATE_COL = "_lb_ref_date"


def prepare_for_bbr(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out[ROW_ID_COL] = np.arange(len(out))
    years = pd.to_numeric(out["year"], errors="coerce")
    out[TARGET_DATE_COL] = pd.to_datetime(
        {"year": years, "month": 7, "day": 1},
        utc=True,
        errors="coerce",
    )
    return out


def merge_pick_bitemporal(left, right, *, left_on, right_on, prefix,
                          id_col="id_lokalId", area_cols=()):
    """Keep one BBR row per observation; reject gaps in validity periods.

    Priority: valid on 1 July; earliest row before the first period;
    latest closed row after the final period. Area breaks remaining ties.
    """
    out = left.copy()
    count_col = prefix + "candidate_rows_n"
    right_columns = list(right.columns) if right is not None else []
    if right is None or right.empty or left_on not in out:
        for column in right_columns:
            out[prefix + column] = pd.NA
        out[count_col] = 0
        out[prefix + "temporal_selection_rule"] = "no_candidate"
        return out

    required = {right_on, id_col, "virkningFra", "virkningTil"}
    if not required.issubset(right.columns):
        raise ValueError(f"BBR is missing columns: {sorted(required - set(right.columns))}")

    keys = out[[ROW_ID_COL, TARGET_DATE_COL, left_on]].copy()
    keys[left_on] = normalize_id(keys[left_on])
    candidates = right.copy()
    candidates[right_on] = normalize_id(candidates[right_on])
    candidates[id_col] = normalize_id(candidates[id_col])
    candidates = candidates.dropna(subset=[right_on, id_col])
    joined = keys.dropna(subset=[left_on]).merge(
        candidates, left_on=left_on, right_on=right_on, how="inner",
    )

    counts = joined.groupby(ROW_ID_COL).size()
    target = joined[TARGET_DATE_COL]
    start = pd.to_datetime(joined["virkningFra"], utc=True, format="mixed", errors="raise")
    end = pd.to_datetime(joined["virkningTil"], utc=True, format="mixed", errors="raise")
    if (start.notna() & end.notna() & start.ge(end)).any():
        raise ValueError("BBR contains an invalid validity interval.")
    group = joined[ROW_ID_COL]
    valid = target.notna() & start.notna() & start.le(target) & (end.isna() | target.lt(end))
    before = target.notna() & target.lt(start.groupby(group).transform("min"))
    all_closed = end.notna().groupby(group).transform("all")
    after = target.notna() & all_closed & target.ge(end.groupby(group).transform("max"))
    missing_date = target.isna()

    joined["_case"] = np.select([valid, before, after, missing_date], [0, 1, 2, 3], default=9)
    joined["_selection_rule"] = np.select(
        [valid, before, after, missing_date],
        ["valid_at_lokalebasen_date",
         "before_first_virkningFra_choose_min_virkningFra",
         "after_last_virkningTil_choose_max_virkningTil",
         "missing_lokalebasen_date_choose_latest_virkningFra"],
        default="candidate_without_temporal_rule",
    )
    joined["_valid_start"] = start.where(valid | missing_date)
    joined["_before_start"] = start.where(before)
    joined["_after_end"] = end.where(after)
    areas = [pd.to_numeric(joined[c], errors="coerce") for c in area_cols if c in joined]
    joined["_area"] = pd.concat(areas, axis=1).max(axis=1) if areas else -1

    best = joined.loc[joined["_case"].lt(9)].sort_values(
        [ROW_ID_COL, "_case", "_valid_start", "_before_start", "_after_end", "_area", id_col],
        ascending=[True, True, False, True, False, False, True], kind="stable",
    ).drop_duplicates(ROW_ID_COL)
    best = best[[ROW_ID_COL, "_selection_rule"] + right_columns].rename(
        columns={"_selection_rule": prefix + "temporal_selection_rule",
                 **{c: prefix + c for c in right_columns}},
    )
    out = out.merge(best, on=ROW_ID_COL, how="left", validate="one_to_one", sort=False)
    out[count_col] = out[ROW_ID_COL].map(counts).fillna(0).astype(int)
    out[prefix + "temporal_selection_rule"] = out[prefix + "temporal_selection_rule"].fillna(
        "no_eligible_temporal_candidate"
    )
    return out

def load_relevant_bbr_enhed(df_dar) -> pd.DataFrame:
    if not BBR_ENHED_PATH.exists():
        raise FileNotFoundError(f"Could not find BBR Enhed: {BBR_ENHED_PATH}")

    wanted = [
        "id_lokalId",
        "adresseIdentificerer",
        "bygning",
        "virkningFra",
        "virkningTil",
        "enh020EnhedensAnvendelse",
        "enh026EnhedensSamledeAreal",
        "enh028ArealTilErhverv",
    ]
    available = set(pd.read_csv(BBR_ENHED_PATH, nrows=0).columns)
    usecols = [column for column in wanted if column in available]
    required = {"id_lokalId", "adresseIdentificerer", "bygning"}
    missing = required - set(usecols)
    if missing:
        raise ValueError(f"BBR Enhed is missing columns: {sorted(missing)}")

    needed = set(normalize_id(df_dar["adresse_id"]).dropna())
    parts = []
    scanned = 0
    for chunk in pd.read_csv(
        BBR_ENHED_PATH,
        usecols=usecols,
        dtype=str,
        low_memory=False,
        chunksize=BBR_CHUNK_SIZE,
    ):
        scanned += len(chunk)
        keep = normalize_id(chunk["adresseIdentificerer"]).isin(needed)
        if keep.any():
            parts.append(chunk.loc[keep].copy())
    out = (
        pd.concat(parts, ignore_index=True)
        if parts
        else pd.DataFrame(columns=usecols)
    )

    print("\n--- Relevant BBR Enhed ---")
    print("Scanned rows:", f"{scanned:,}")
    print("Relevant rows:", f"{len(out):,}")
    return out


def load_relevant_bbr_bygning(df_dar, bbr_enhed) -> pd.DataFrame:
    """
    Stream the source and build the relevant subset in memory.

    There is deliberately no second cache file: each program run derives this
    subset from the current DAR results and current BBR source.
    """
    if not BBR_BYGNING_PATH.exists():
        raise FileNotFoundError(
            f"Could not find BBR Bygning: {BBR_BYGNING_PATH}"
        )

    needed_access_ids = set(
        normalize_id(df_dar["adgangsadresseid"]).dropna()
    )
    needed_building_ids = set(
        normalize_id(bbr_enhed["bygning"]).dropna()
    )

    wanted = [
        "id_lokalId",
        "kommunekode",
        "status",
        "registreringFra",
        "registreringTil",
        "virkningFra",
        "virkningTil",
        "byg007Bygningsnummer",
        "byg021BygningensAnvendelse",
        "byg026Opførelsesår",
        "byg027OmTilbygningsår",
        "byg038SamletBygningsareal",
        "byg039BygningensSamledeBoligAreal",
        "byg040BygningensSamledeErhvervsAreal",
        "byg041BebyggetAreal",
        "byg054AntalEtager",
        "byg056Varmeinstallation",
        "byg057Opvarmningsmiddel",
        "byg404Koordinat",
        "byg406Koordinatsystem",
        "jordstykke",
        "husnummer",
        "grund",
    ]
    available = set(pd.read_csv(BBR_BYGNING_PATH, nrows=0).columns)
    usecols = [column for column in wanted if column in available]
    required = {"id_lokalId", "husnummer", "kommunekode", "status"}
    missing = required - set(usecols)
    if missing:
        raise ValueError(
            f"BBR Bygning is missing strict-validation columns: {sorted(missing)}"
        )

    parts = []
    scanned = 0
    for chunk in pd.read_csv(
        BBR_BYGNING_PATH,
        usecols=usecols,
        dtype=str,
        low_memory=False,
        chunksize=BBR_CHUNK_SIZE,
    ):
        scanned += len(chunk)
        access_key = normalize_id(chunk["husnummer"])
        building_key = normalize_id(chunk["id_lokalId"])
        keep = (
            access_key.isin(needed_access_ids)
            | building_key.isin(needed_building_ids)
        )
        if keep.any():
            parts.append(chunk.loc[keep].copy())

    out = (
        pd.concat(parts, ignore_index=True)
        if parts
        else pd.DataFrame(columns=usecols)
    )
    print("\n--- Relevant BBR Bygning (built fresh, not cached) ---")
    print("Scanned rows:", f"{scanned:,}")
    print("Relevant rows:", f"{len(out):,}")
    return out


# =============================================================================
# STRICT DIRECT BUILDING VALIDATION
# =============================================================================

def merge_direct_building_strict(left, buildings):
    """
    Accept adgangsadresseid -> BBR Bygning only when:
      - no building was found through BBR Enhed;
      - the access ID maps to exactly one distinct building;
      - the DAR match itself passed street/house/postcode validation;
      - DAR and BBR municipality codes are present and equal;
      - BBR status is 6;
      - the BBR row is valid on the observation date.
    """
    out = left.copy()
    prefix = "bbr_bygning_direct__"
    enhed_building_col = "bbr_bygning_from_enhed__id_lokalId"
    enhed_found = (
        out[enhed_building_col].notna()
        if enhed_building_col in out
        else pd.Series(False, index=out.index)
    )
    has_access = out["adgangsadresseid"].notna()
    dar_validated = (
        as_bool(out["validation_ok"])
        if "validation_ok" in out
        else pd.Series(False, index=out.index)
    )
    needed = ~enhed_found & has_access & dar_validated

    join_col = "_direct_access_id"
    out[join_col] = out["adgangsadresseid"].where(needed)

    if (
        buildings is not None
        and not buildings.empty
        and {"husnummer", "id_lokalId"}.issubset(buildings.columns)
    ):
        keys = pd.DataFrame({
            "access": normalize_id(buildings["husnummer"]),
            "building": normalize_id(buildings["id_lokalId"]),
        }).dropna()
        distinct = keys.groupby("access")["building"].nunique()
    else:
        distinct = pd.Series(dtype="int64")

    out["bbr_direct_candidate_buildings_n"] = (
        normalize_id(out[join_col])
        .map(distinct)
        .fillna(0)
        .astype(int)
    )

    out = merge_pick_bitemporal(
        out,
        buildings,
        left_on=join_col,
        right_on="husnummer",
        prefix=prefix,
        id_col="id_lokalId",
        area_cols=[
            "byg040BygningensSamledeErhvervsAreal",
            "byg038SamletBygningsareal",
            "byg041BebyggetAreal",
        ],
    )

    right_columns = list(buildings.columns) if buildings is not None else []
    for column in right_columns:
        target = prefix + column
        if target not in out:
            out[target] = pd.NA

    id_col = prefix + "id_lokalId"
    municipality_col = prefix + "kommunekode"
    status_col = prefix + "status"
    temporal_col = prefix + "temporal_selection_rule"
    for column in [id_col, municipality_col, status_col]:
        if column not in out:
            out[column] = pd.NA
    if temporal_col not in out:
        out[temporal_col] = "no_candidate"

    candidate_found = out[id_col].notna()
    unique = out["bbr_direct_candidate_buildings_n"].eq(1)
    dar_municipality = normalize_kommunekode(out["kommunekode"])
    bbr_municipality = normalize_kommunekode(out[municipality_col])
    municipality_available = (
        dar_municipality.notna() & bbr_municipality.notna()
    )
    municipality_ok = (
        municipality_available
        & dar_municipality.eq(bbr_municipality)
    ).fillna(False)
    status = (
        out[status_col]
        .astype("string")
        .str.strip()
        .str.replace(r"\.0+$", "", regex=True)
    )
    status_available = status.notna()
    status_ok = status.eq("6").fillna(False)
    temporal_ok = out[temporal_col].eq(
        "valid_at_lokalebasen_date"
    ).fillna(False)

    accepted = (
        needed
        & candidate_found
        & unique
        & municipality_ok
        & status_ok
        & temporal_ok
    ).fillna(False)

    out["bbr_direct_candidate_id"] = out[id_col]
    out["bbr_direct_kommunekode_ok"] = municipality_ok
    out["bbr_direct_status_ok"] = status_ok
    out["bbr_direct_temporal_rule_ok"] = temporal_ok
    out["bbr_direct_building_accepted"] = accepted
    out["bbr_direct_building_ambiguous"] = (
        needed & out["bbr_direct_candidate_buildings_n"].gt(1)
    )
    out["bbr_direct_validation"] = np.select(
        [
            enhed_found,
            ~has_access,
            ~dar_validated,
            out["bbr_direct_candidate_buildings_n"].eq(0),
            out["bbr_direct_candidate_buildings_n"].gt(1),
            ~candidate_found,
            ~municipality_available,
            ~municipality_ok,
            ~status_available,
            ~status_ok,
            ~temporal_ok,
            accepted,
        ],
        [
            "not_used_enhed_building_available",
            "rejected_missing_adgangsadresseid",
            "rejected_dar_match_not_component_validated",
            "rejected_no_bbr_building_candidate",
            "rejected_multiple_bbr_buildings",
            "rejected_no_temporal_candidate",
            "rejected_missing_kommunekode",
            "rejected_kommunekode_mismatch",
            "rejected_missing_bbr_status",
            "rejected_bbr_status_not_6",
            "rejected_not_valid_at_lokalebasen_date",
            "accepted_unique_building",
        ],
        default="rejected_unspecified",
    )

    direct_data_columns = [
        prefix + column
        for column in right_columns
        if prefix + column in out
    ]
    if direct_data_columns:
        out.loc[~accepted, direct_data_columns] = pd.NA

    out["bbr_direct_unique_buildings_n"] = (
        out["bbr_direct_candidate_buildings_n"]
    )
    return out.drop(columns=[join_col], errors="ignore")


def combine_buildings(df):
    out = df.copy()
    from_prefix = "bbr_bygning_from_enhed__"
    direct_prefix = "bbr_bygning_direct__"
    final_prefix = "bbr_bygning_final__"

    bases = {
        column[len(from_prefix):]
        for column in out
        if column.startswith(from_prefix)
    } | {
        column[len(direct_prefix):]
        for column in out
        if column.startswith(direct_prefix)
    }
    from_id = from_prefix + "id_lokalId"
    direct_id = direct_prefix + "id_lokalId"
    use_enhed = (
        out[from_id].notna()
        if from_id in out
        else pd.Series(False, index=out.index)
    )
    use_direct = (
        ~use_enhed & out[direct_id].notna()
        if direct_id in out
        else pd.Series(False, index=out.index)
    )

    for base in bases:
        from_column = from_prefix + base
        direct_column = direct_prefix + base
        if from_column in out and direct_column in out:
            out[final_prefix + base] = (
                out[from_column]
                .where(use_enhed)
                .combine_first(out[direct_column].where(use_direct))
            )
        elif from_column in out:
            out[final_prefix + base] = out[from_column].where(use_enhed)
        else:
            out[final_prefix + base] = out[direct_column].where(use_direct)
    return out


# =============================================================================
# GIS COORDINATES AND SIMPLE ROUTE LABELS
# =============================================================================

def add_gis_coordinates(df):
    """Use a valid BBR coordinate pair, otherwise a valid DAR pair."""
    from pyproj import Transformer

    out = df.copy()
    coordinate = out.get("bbr_bygning_final__byg404Koordinat", pd.Series(index=out.index, dtype="object"))
    parsed = coordinate.apply(parse_point_to_xy)
    if out.empty:
        parsed = pd.DataFrame(index=out.index, columns=["GISX_bbr", "GISY_bbr"], dtype=float)
    out[["GISX_bbr", "GISY_bbr"]] = parsed

    bbr_present = out["GISX_bbr"].notna() & out["GISY_bbr"].notna()
    out["bbr_coordinate_valid"] = (
        out["GISX_bbr"].between(350_000, 950_000)
        & out["GISY_bbr"].between(6_000_000, 6_500_000)
    )
    out["bbr_coordinate_rejected"] = bbr_present & ~out["bbr_coordinate_valid"]
    out.loc[~out["bbr_coordinate_valid"], ["GISX_bbr", "GISY_bbr"]] = np.nan

    out["GISX_dar"] = pd.to_numeric(out["x"], errors="coerce")
    out["GISY_dar"] = pd.to_numeric(out["y"], errors="coerce")
    lonlat = out["GISX_dar"].between(7, 16) & out["GISY_dar"].between(54, 58)
    if lonlat.any():
        transformer = Transformer.from_crs(4326, 25832, always_xy=True)
        x, y = transformer.transform(out.loc[lonlat, "GISX_dar"].to_numpy(), out.loc[lonlat, "GISY_dar"].to_numpy())
        out.loc[lonlat, "GISX_dar"] = x
        out.loc[lonlat, "GISY_dar"] = y

    out["dar_coordinate_valid"] = (
        out["GISX_dar"].between(350_000, 950_000)
        & out["GISY_dar"].between(6_000_000, 6_500_000)
    )
    out.loc[~out["dar_coordinate_valid"], ["GISX_dar", "GISY_dar"]] = np.nan
    out["GISX"] = out["GISX_bbr"].combine_first(out["GISX_dar"])
    out["GISY"] = out["GISY_bbr"].combine_first(out["GISY_dar"])
    out["GIS_source"] = np.select(
        [out["bbr_coordinate_valid"], out["dar_coordinate_valid"]],
        ["BBR Bygning byg404Koordinat", "DAR x/y"], default="missing",
    )
    out["GISX_dawa"] = out["GISX_dar"]
    out["GISY_dawa"] = out["GISY_dar"]
    out["dawa_coordinate_valid"] = out["dar_coordinate_valid"]
    return out

def add_join_path(df):
    out = df.copy()
    has_gis = out["GISX"].notna() & out["GISY"].notna()
    has_dar = out["adgangsadresseid"].notna()
    via_enhed = out["has_bbr_bygning_from_enhed"]
    via_direct = out["bbr_direct_building_used_as_final"]

    out["join_path"] = np.select(
        [
            via_enhed & has_gis,
            via_direct & has_gis,
            has_dar & has_gis,
            via_enhed,
            via_direct,
            has_dar,
        ],
        [
            "adresse_id -> BBR Enhed -> BBR Bygning -> GIS",
            "adgangsadresseid -> BBR Bygning -> GIS",
            "DAR match -> GIS",
            "adresse_id -> BBR Enhed -> BBR Bygning -> no GIS coordinates",
            "adgangsadresseid -> BBR Bygning -> no GIS coordinates",
            "DAR match -> no GIS coordinates",
        ],
        default="No DAR match -> no GIS coordinates",
    )
    out["bbr_match_level"] = out["join_path"]
    return out


# =============================================================================
# PIPELINE
# =============================================================================

def build_slim_output(df, origin_columns):
    wanted = list(origin_columns) + [
        # DAR
        "matched_address",
        "match_level",
        "match_method",
        "method_order",
        "query_used",
        "lookup_status",
        "lookup_error",
        "lookup_version",
        "adresse_id",
        "adgangsadresseid",
        "kommunekode",
        "postnr",
        "postnrnavn",
        "vejnavn",
        "husnr",
        "match_score",
        "validation_ok",
        "validation_reason",
        "required_postnr",
        "required_vejnavn",
        "required_husnr_values",
        "candidate_postnr_ok",
        "candidate_vejnavn_ok",
        "candidate_husnr_ok",
        "candidates_checked_n",
        "candidates_rejected_n",
        "methods_tried_n",
        "api_failures_n",
        "detail_complete",

        # GIS
        "GISX",
        "GISY",
        "GIS_source",
        "GISX_bbr",
        "GISY_bbr",
        "GISX_dar",
        "GISY_dar",
        "bbr_coordinate_valid",
        "bbr_coordinate_rejected",
        "dar_coordinate_valid",
        "GISX_dawa",
        "GISY_dawa",
        "dawa_coordinate_valid",

        # BBR Enhed
        "has_bbr_enhed",
        "bbr_enhed__temporal_selection_rule",
        "bbr_enhed__id_lokalId",
        "bbr_enhed__adresseIdentificerer",
        "bbr_enhed__bygning",
        "bbr_enhed__enh020EnhedensAnvendelse",
        "bbr_enhed__enh026EnhedensSamledeAreal",
        "bbr_enhed__enh028ArealTilErhverv",

        # Final BBR Bygning
        "has_bbr_bygning_final",
        "bbr_bygning_final__temporal_selection_rule",
        "bbr_bygning_final__id_lokalId",
        "bbr_bygning_final__husnummer",
        "bbr_bygning_final__kommunekode",
        "bbr_bygning_final__status",
        "bbr_bygning_final__byg007Bygningsnummer",
        "bbr_bygning_final__byg021BygningensAnvendelse",
        "bbr_bygning_final__byg026Opførelsesår",
        "bbr_bygning_final__byg027OmTilbygningsår",
        "bbr_bygning_final__byg038SamletBygningsareal",
        "bbr_bygning_final__byg039BygningensSamledeBoligAreal",
        "bbr_bygning_final__byg040BygningensSamledeErhvervsAreal",
        "bbr_bygning_final__byg041BebyggetAreal",
        "bbr_bygning_final__byg054AntalEtager",
        "bbr_bygning_final__byg056Varmeinstallation",
        "bbr_bygning_final__byg057Opvarmningsmiddel",
        "bbr_bygning_final__byg404Koordinat",
        "bbr_bygning_final__byg406Koordinatsystem",

        # Diagnostics
        "join_path",
        "has_bbr_bygning_from_enhed",
        "has_bbr_bygning_direct",
        "bbr_bygning_from_enhed__temporal_selection_rule",
        "bbr_bygning_direct__temporal_selection_rule",
        "bbr_direct_building_used_as_final",
        "bbr_direct_validation",
        "bbr_direct_candidate_id",
        "bbr_direct_candidate_buildings_n",
        "bbr_direct_kommunekode_ok",
        "bbr_direct_status_ok",
        "bbr_direct_temporal_rule_ok",
        "bbr_direct_building_accepted",
        "bbr_direct_unique_buildings_n",
        "bbr_direct_building_ambiguous",
    ]
    wanted = list(dict.fromkeys(wanted))
    return df[[column for column in wanted if column in df]].copy()


def main():
    PROCESSED_FOLDER.mkdir(parents=True, exist_ok=True)

    # 1. Read yearly rents, classifications, facilities and CPI.
    origins = load_lokalebasen()
    origin_columns = list(origins.columns)

    # 2. Resolve each distinct address once.
    dar_cache = lookup_many_dar(origins["address_clean"])
    df = origins.merge(dar_cache, on="address_clean", how="left", validate="many_to_one")

    # 3. Read only relevant BBR rows, in chunks.
    enheder = load_relevant_bbr_enhed(df)
    bygninger = load_relevant_bbr_bygning(df, enheder)
    df = prepare_for_bbr(df)

    # 4. Address -> Enhed -> Bygning.
    df = merge_pick_bitemporal(
        df, enheder, left_on="adresse_id", right_on="adresseIdentificerer",
        prefix="bbr_enhed__", area_cols=["enh028ArealTilErhverv", "enh026EnhedensSamledeAreal"],
    )
    for column in ["bbr_enhed__id_lokalId", "bbr_enhed__bygning"]:
        if column not in df:
            df[column] = pd.NA
    df = merge_pick_bitemporal(
        df, bygninger, left_on="bbr_enhed__bygning", right_on="id_lokalId",
        prefix="bbr_bygning_from_enhed__",
        area_cols=["byg040BygningensSamledeErhvervsAreal", "byg038SamletBygningsareal", "byg041BebyggetAreal"],
    )

    # 5. If no building was found, try one strictly validated access-address match.
    df = merge_direct_building_strict(df, bygninger)
    df = combine_buildings(df)
    for name in ["enhed", "bygning_from_enhed", "bygning_direct", "bygning_final"]:
        column = f"bbr_{name}__id_lokalId"
        if column not in df:
            df[column] = pd.NA
        df[f"has_bbr_{name}"] = df[column].notna()
    df["bbr_direct_building_used_as_final"] = (
        df["has_bbr_bygning_direct"] & ~df["has_bbr_bygning_from_enhed"]
    )

    # 6. Add coordinates and save both outputs.
    df = add_join_path(add_gis_coordinates(df))
    if len(df) != len(origins) or df["id"].duplicated().any():
        raise AssertionError("DAR/BBR joins changed row count or duplicated an observation.")
    full = df.drop(columns=[ROW_ID_COL, TARGET_DATE_COL])
    slim = build_slim_output(full, origin_columns)
    full.to_csv(OUT_FULL_PATH, index=False)
    slim.to_csv(OUT_GIS_PATH, index=False)
    print("\nJoin paths:")
    print(full["join_path"].value_counts(dropna=False))
    print("\nSaved:", OUT_FULL_PATH, OUT_GIS_PATH, sep="\n")
    return full, slim


full_output, gis_output = main()
try:
    display(gis_output.head(50))
    display(gis_output)
except NameError:
    print("\nPreview of slim GIS dataset:")
    print(gis_output.head(50))

# %%

# gammel

In [ ]:
def count_rows_in_csv(file_path):
    with open(file_path, "rb") as f:
        return sum(chunk.count(b"\n") for chunk in iter(lambda: f.read(1024*1024), b""))

def filtrer_bbr_csv_OPFØRT_a_AktuelleHistoriskeData(input_folder, input_filename, output_folder, output_filename,
                                                    status=6, cols=None, hovedkat=None, chunksize=2_000_000):
    """
    Filtrerer BBR data for at beholde kun de bygninger der er "opført"
    og stadig relevante (dvs. registreringTil er NaN)
    """

    total_chunks = count_rows_in_csv(f"{input_folder}/{input_filename}") / chunksize + 1 # bruges udelukkende til tqdm kan tælle progress
    count_fjernet_ialt = 0
    first_write = True
    
    for i, chunk in enumerate(tqdm(pd.read_csv(f"{input_folder}/{input_filename}", chunksize=chunksize, usecols=cols),total=total_chunks)):

        mask_statusOpført = chunk["status"] == status
        mask_stadigrelevant = chunk['registreringTil'].isna()

        count_fjernet_ialt += sum(~mask_stadigrelevant)

        filtered = chunk[mask_stadigrelevant & mask_statusOpført]

        # fjerner forstyrrende Z'ere, som guderne må vide hvorfor er der i nogle af datokolonnerne.
        for dim in ['byg406Koordinatsystem', 'registreringFra', 'virkningFra', 'virkningTil']:
            if dim in chunk.columns:
                chunk[dim] = chunk[dim].astype(str).str.replace("Z","")

        filtered.to_csv(
            f"{output_folder}/{output_filename}",
            mode="w" if first_write else "a",
            index=False,
            header=first_write
        )
        first_write = False

    print("Done!")

: 

In [4]:
def count_rows_in_csv(file_path):
    with open(file_path, "rb") as f:
        return sum(chunk.count(b"\n") for chunk in iter(lambda: f.read(1024*1024), b""))

def filtrer_bbr_csv_OPFØRT_a_AktuelleHistoriskeData(
    input_folder, input_filename, output_folder, output_filename, status=6, cols=None,
    hovedkat=None, chunksize=2_000_000, overwrite_coordinate=True,
    overwrite_construction_year=True, id_col="id_lokalId"
):
    """
    Filtrerer BBR data for at beholde kun de bygninger der er "opført"
    og stadig relevante (dvs. registreringTil er NaN)

    overwrite_coordinate=True:
        Overskriver kun koordinater på rækker med dårligere datakvalitet, når
        der findes én entydig koordinat med højere datakvalitet inden for id'et.
        Ved samme bedste kvalitet og forskellige koordinater ændres intet.

    overwrite_construction_year=True:
        Overskriver opførelsesåret inden for hvert id med værdien fra rækken
        med nyeste virkningFra. Det numerisk højeste opførelsesår prioriteres ikke.
    """

    input_path = f"{input_folder}/{input_filename}"
    total_chunks = count_rows_in_csv(input_path) / chunksize + 1 # bruges udelukkende til tqdm kan tælle progress
    count_fjernet_ialt, first_write = 0, True

    # Tilføjer nødvendige hjælpekolonner ved overskrivning, men beholder kun cols i outputtet
    read_cols = cols
    if cols is not None and (overwrite_coordinate or overwrite_construction_year):
        required = [id_col, "status", "registreringTil", "virkningFra"]
        if overwrite_coordinate: required += ["byg404Koordinat", "byg134KvalitetAfKoordinatsæt"]
        if overwrite_construction_year: required += ["byg026Opførelsesår"]
        read_cols = list(dict.fromkeys(list(cols) + required))

    best_coordinates = best_coordinate_quality = best_construction_years = None

    # Første gennemlæsning: finder den bedste værdi på tværs af alle chunks
    if overwrite_coordinate or overwrite_construction_year:
        coordinate_candidates, construction_year_candidates = [], []

        for chunk in tqdm(pd.read_csv(input_path, chunksize=chunksize, usecols=read_cols),
                          total=total_chunks, desc="Finder bedste værdier"):

            # fjerner forstyrrende Z'ere før datoerne eventuelt konverteres
            for dim in ["registreringFra", "registreringTil", "virkningFra", "virkningTil"]:
                if dim in chunk: chunk[dim] = chunk[dim].astype("string").str.replace("Z", "", regex=False)

            filtered = chunk[chunk["registreringTil"].isna() & chunk["status"].eq(status)].copy()
            filtered["_virkningFra"] = pd.to_datetime(filtered["virkningFra"], errors="coerce")

            # Gemmer forskellige koordinater og kvaliteter, så lighed i kvalitet ikke afgøres efter virkningFra
            if overwrite_coordinate:
                x = filtered.loc[filtered["byg404Koordinat"].notna(),
                                 [id_col, "byg404Koordinat", "byg134KvalitetAfKoordinatsæt"]].copy()
                x["_quality"] = pd.to_numeric(x["byg134KvalitetAfKoordinatsæt"], errors="coerce")
                coordinate_candidates.append(
                    x.dropna(subset=["_quality"]).drop_duplicates([id_col, "_quality", "byg404Koordinat"])
                )

            # Opførelsesåret fra rækken med nyeste virkningFra vælges
            if overwrite_construction_year:
                x = filtered.loc[
                    filtered["byg026Opførelsesår"].notna() & filtered["_virkningFra"].notna(),
                    [id_col, "byg026Opførelsesår", "_virkningFra"]
                ]
                construction_year_candidates.append(
                    x.sort_values([id_col, "_virkningFra"], ascending=[True, False],
                                  kind="mergesort").drop_duplicates(id_col)
                )

        # Vælger kun koordinat, hvis den bedste kvalitet har præcis én forskellig koordinat
        if overwrite_coordinate and coordinate_candidates:
            x = pd.concat(coordinate_candidates, ignore_index=True).drop_duplicates(
                [id_col, "_quality", "byg404Koordinat"]
            )
            x = x[x["_quality"].eq(x.groupby(id_col)["_quality"].transform("min"))]
            x = x[x.groupby(id_col)["byg404Koordinat"].transform("nunique").eq(1)].drop_duplicates(id_col)
            best_coordinates = x.set_index(id_col)["byg404Koordinat"]
            best_coordinate_quality = x.set_index(id_col)["_quality"]

        if overwrite_construction_year and construction_year_candidates:
            x = pd.concat(construction_year_candidates, ignore_index=True)
            best_construction_years = (
                x.sort_values([id_col, "_virkningFra"], ascending=[True, False], kind="mergesort")
                 .drop_duplicates(id_col).set_index(id_col)["byg026Opførelsesår"]
            )

    for i, chunk in enumerate(tqdm(pd.read_csv(input_path, chunksize=chunksize, usecols=read_cols),
                                   total=total_chunks, desc="Filtrerer og gemmer")):

        # fjerner forstyrrende Z'ere uden at omdanne manglende værdier til teksten "nan"
        for dim in ["registreringFra", "registreringTil", "virkningFra", "virkningTil"]:
            if dim in chunk: chunk[dim] = chunk[dim].astype("string").str.replace("Z", "", regex=False)

        mask_statusOpført = chunk["status"] == status
        mask_stadigrelevant = chunk["registreringTil"].isna()
        count_fjernet_ialt += sum(~mask_stadigrelevant)
        filtered = chunk[mask_stadigrelevant & mask_statusOpført].copy()

        # Overskriver kun koordinater med strengt dårligere eller manglende datakvalitet
        if overwrite_coordinate and best_coordinates is not None:
            best_coord = filtered[id_col].map(best_coordinates)
            best_quality = filtered[id_col].map(best_coordinate_quality)
            current_quality = pd.to_numeric(filtered["byg134KvalitetAfKoordinatsæt"], errors="coerce")
            replace = best_coord.notna() & best_quality.notna() & (
                current_quality.isna() | current_quality.gt(best_quality)
            )
            filtered.loc[replace, "byg404Koordinat"] = best_coord.loc[replace]

        # Overskriver alle rækker for hvert id med opførelsesåret fra nyeste virkningFra
        if overwrite_construction_year and best_construction_years is not None:
            filtered["byg026Opførelsesår"] = (
                filtered[id_col].map(best_construction_years).combine_first(filtered["byg026Opførelsesår"])
            )

        # fjerner forstyrrende Z'ere, som guderne må vide hvorfor er der i nogle af datokolonnerne.
        if "byg406Koordinatsystem" in filtered:
            filtered["byg406Koordinatsystem"] = (
                filtered["byg406Koordinatsystem"].astype("string").str.replace("Z", "", regex=False)
            )

        # Fjerner hjælpekolonner igen, hvis cols oprindeligt var angivet
        if cols is not None: filtered = filtered[[c for c in cols if c in filtered.columns]]

        filtered.to_csv(
            f"{output_folder}/{output_filename}",
            mode="w" if first_write else "a",
            index=False,
            header=first_write
        )
        first_write = False

    print("Done!")

In [6]:
input_folder = "D:/tung_databehandling/BBR/raw" # data_folder_path_output = "P:/cmo/TFP&Tæthed/BBR/data"
input_filename = "BBR_V3_Bygning_TotalDownload_csv_Bitemporal_653.csv"
output_folder = "D:/tung_databehandling/BBR/raw"
output_filename = "BBR_V3_Bygning_TotalDownload_csv_Bitemporal_653_registreringTil_isna_Status6_fixedKoordinater_fixedOpførelsesår_selected_cols.csv"

cols_to_extract = ['id_namespace', # identifikation af et namespace inden for hvilket lokalId er unik
 'id_lokalId', # identifikation af objektet
 'kommunekode', # se kodeliste: https://teknik.bbr.dk/kodelister/0/1/0/Kommunekode
 'registreringFra',
 'registreringTil',
 'virkningFra', # tidspunktet hvor virkningen af den pågældende version af bygværkselementet er startet
 'virkningTil', # alle uden værdi (NaN)
 'status', # kode for bygværkselementets status i den pågældende version, dvs. elementets tilstand i den samlede livscyklus
 'byg021BygningensAnvendelse', # se kodeliste: https://teknik.bbr.dk/kodelister/0/1/0/BygAnvendelse
 'byg026Opførelsesår', # angiver bygningens opførselsår
 'byg027OmTilbygningsår', # Bygningens seneste om- eller tilbygningsår. Angives i formatet åååå
 'byg038SamletBygningsareal', # Det samlede bygningsareal er summen af etagernes arealer, men ekskl. kælderareal og areal af tagetage. Arealerne måles til ydersiden af ydervæggene (bruttoetageareal). Værdisæt 0 – 300.000 (kan være negativ ved byggesagsdata)
 'byg039BygningensSamledeBoligAreal', # Areal, der anvendes til boligformål, !!!herunder også udnyttet areal af tagetage og lovlig beboelse i kælderetage!!!. Arealet måles til ydersiden af ydervægge (bruttoetageareal). Arealet er inkl. evt. andel af adgangsarealer. Værdisæt 0 – 300.000 (kan være negativ ved byggesagsdata)
 'byg040BygningensSamledeErhvervsAreal', # Areal, der udelukkende anvendes til erhverv (ikke-boligformål), herunder også udnyttet areal af tagetage og kælderetage. Arealet måles til ydersiden af ydervægge (bruttoetageareal). Arealet er inkl. evt. andel af adgangsarealer. Værdisæt 0 – 300.000 (kan være negativ ved byggesagsdata)
 'byg041BebyggetAreal', # Arealet registreres som stueetagens areal målt til ydersiden af ydervæggene. For bygninger, hvor etagerne arealmæssigt afviger fra hinanden, angives arealet af den største etage. Kælderetagen registreres særskilt i felt ETA.20 i ''kælder''. Arealet af åbne, overdækkede terrasser, åbne altaner og åbne indgangspartier medregnes ikke. Såfremt sådanne arealer lukkes, skal arealet medregnes. Arealer af porte, luftsluser o. lign. medtages til den linie, som bygningens ydervægge iøvrigt angiver. Arealet af garage, carport eller udhus, der udgør en del af bygningen (indenfor bygningens ydervægge og under samme tag) medregnes ligeledes. Værdisæt 0 – 300.000 (kan være negativ ved byggesagsdata)
 'byg054AntalEtager',
 #'byg070Fredning', # personligt nysgerrig hvor mange der er og hvad deres karakteristika er:) - ikke relevant for agglomerationsprojektet
 'byg134KvalitetAfKoordinatsæt', # kodeliste der angiver kvaliteten af geometrioplysninger (1: Sikker, 2(udfases): næsten sikker, 3: usikker)
 'byg404Koordinat', # Angiver bygningens geografiske repræsentation i form af et punkt
 'byg406Koordinatsystem'] # burde bruge 5 = UTM 32 Euref89 (= epsg:25832)

filtrer_bbr_csv_OPFØRT_a_AktuelleHistoriskeData(
    input_folder, input_filename, output_folder, output_filename, status=6, cols=cols_to_extract,
    hovedkat=None, chunksize=2_000_000, overwrite_coordinate=True,
    overwrite_construction_year=True, id_col="id_lokalId"
)

Finder bedste værdier:  98%|█████████▊| 23/23.430663 [05:59<00:06, 15.62s/it]
Filtrerer og gemmer:  98%|█████████▊| 23/23.430663 [09:18<00:10, 24.27s/it]


Done!


In [ ]:
# filtrer BBRbygning
# da det er meget hurtigere at behandle data på min lokale ssd bruges dette.
input_folder = "D:/tung_databehandling/BBR/raw" # data_folder_path_output = "P:/cmo/TFP&Tæthed/BBR/data"
input_filename = "BBR_V3_Bygning_TotalDownload_csv_Bitemporal_653.csv"
output_folder = "D:/tung_databehandling/BBR/raw"
output_filename = "BBR_V3_Bygning_TotalDownload_csv_Bitemporal_653_registreringTil_isna_Status6.csv"

# # bruges til "BBR_NavKatÆndringer_Beskrivende.ipynb"
# cols_to_extract = ['id_namespace', 'id_lokalId', 'kommunekode', 'registreringFra',
#                    'registreringTil', 'virkningFra', 'virkningTil', 'status',
#                    'byg007Bygningsnummer', 'byg021BygningensAnvendelse',
#                    'byg026Opførelsesår', 'byg038SamletBygningsareal',
#                    'byg041BebyggetAreal', 'byg054AntalEtager', 'byg404Koordinat']

# # bruge stil "BBR_AktueltSnapshot_xxx.ipynb"
# cols_to_extract = ['id_namespace', # identifikation af et namespace inden for hvilket lokalId er unik
#  'id_lokalId', # identifikation af objektet
#  'kommunekode', # se kodeliste: https://teknik.bbr.dk/kodelister/0/1/0/Kommunekode
#  'registreringFra',
#  'registreringTil',
#  'virkningFra', # tidspunktet hvor virkningen af den pågældende version af bygværkselementet er startet
#  'virkningTil', # alle uden værdi (NaN)
#  'status', # kode for bygværkselementets status i den pågældende version, dvs. elementets tilstand i den samlede livscyklus
#  'byg021BygningensAnvendelse', # se kodeliste: https://teknik.bbr.dk/kodelister/0/1/0/BygAnvendelse
#  'byg026Opførelsesår', # angiver bygningens opførselsår
#  'byg027OmTilbygningsår', # Bygningens seneste om- eller tilbygningsår. Angives i formatet åååå
#  'byg038SamletBygningsareal', # Det samlede bygningsareal er summen af etagernes arealer, men ekskl. kælderareal og areal af tagetage. Arealerne måles til ydersiden af ydervæggene (bruttoetageareal). Værdisæt 0 – 300.000 (kan være negativ ved byggesagsdata)
#  'byg039BygningensSamledeBoligAreal', # Areal, der anvendes til boligformål, !!!herunder også udnyttet areal af tagetage og lovlig beboelse i kælderetage!!!. Arealet måles til ydersiden af ydervægge (bruttoetageareal). Arealet er inkl. evt. andel af adgangsarealer. Værdisæt 0 – 300.000 (kan være negativ ved byggesagsdata)
#  'byg040BygningensSamledeErhvervsAreal', # Areal, der udelukkende anvendes til erhverv (ikke-boligformål), herunder også udnyttet areal af tagetage og kælderetage. Arealet måles til ydersiden af ydervægge (bruttoetageareal). Arealet er inkl. evt. andel af adgangsarealer. Værdisæt 0 – 300.000 (kan være negativ ved byggesagsdata)
#  'byg041BebyggetAreal', # Arealet registreres som stueetagens areal målt til ydersiden af ydervæggene. For bygninger, hvor etagerne arealmæssigt afviger fra hinanden, angives arealet af den største etage. Kælderetagen registreres særskilt i felt ETA.20 i ''kælder''. Arealet af åbne, overdækkede terrasser, åbne altaner og åbne indgangspartier medregnes ikke. Såfremt sådanne arealer lukkes, skal arealet medregnes. Arealer af porte, luftsluser o. lign. medtages til den linie, som bygningens ydervægge iøvrigt angiver. Arealet af garage, carport eller udhus, der udgør en del af bygningen (indenfor bygningens ydervægge og under samme tag) medregnes ligeledes. Værdisæt 0 – 300.000 (kan være negativ ved byggesagsdata)
#  'byg054AntalEtager',
#  #'byg070Fredning', # personligt nysgerrig hvor mange der er og hvad deres karakteristika er:) - ikke relevant for agglomerationsprojektet
#  'byg134KvalitetAfKoordinatsæt', # kodeliste der angiver kvaliteten af geometrioplysninger (1: Sikker, 2(udfases): næsten sikker, 3: usikker)
#  'byg404Koordinat', # Angiver bygningens geografiske repræsentation i form af et punkt
#  'byg406Koordinatsystem'] # burde bruge 5 = UTM 32 Euref89 (= epsg:25832)

# indlæs blot samtlige kolonner
cols_to_extract = None

filtrer_bbr_csv_OPFØRT_a_AktuelleHistoriskeData(
    input_folder, input_filename, output_folder, output_filename,
    cols=cols_to_extract,
    status=6,
    )

# diagnostics
print("Rækker før:",count_rows_in_csv(f"{input_folder}/{input_filename}"))
print("Rækker efter:", count_rows_in_csv(f"{output_folder}/{output_filename}"))

 98%|█████████▊| 23/23.430663 [10:23<00:11, 27.10s/it]


Done!
Rækker før: 44861326
Rækker efter: 29448317


In [ ]:
# filtrer BBR enhed (præcist samme filter)
input_folder_enhed = r"D:\tung_databehandling\BBR\raw"
input_filename_enhed = "BBR_V3_Enhed_TotalDownload_csv_Bitemporal_653.csv"
output_folder_enhed = r"D:/tung_databehandling/BBR/raw"
output_filename_enhed = "BBR_V3_Enhed_TotalDownload_csv_Bitemporal_653_registreringTil_isna_Status6.csv"

filtrer_bbr_csv_OPFØRT_a_AktuelleHistoriskeData(
    input_folder_enhed, input_filename_enhed, output_folder_enhed, output_filename_enhed,
    cols=cols_to_extract,
    status=6,
)

# diagnostics
print("Rækker før:",count_rows_in_csv(f"{input_folder_enhed}/{input_filename_enhed}"))
print("Rækker efter:", count_rows_in_csv(f"{output_folder_enhed}/{output_filename_enhed}"))